# IEEE 13 barras com vários snapshots por vetor de parâmetros

Para cada vetor $\theta=(r_{11},\dots,x_{33})$ sorteado, o trecho **RG60–632** (linha 650632, 2000 ft) recebe essa impedância e o fluxo de potência é resolvido **K = 5** vezes. Entre os snapshots, cada uma das 15 cargas é multiplicada por um fator independente $U(0{,}7;\,1{,}3)$ (mesmo fator para kW e kvar). Assim $\theta$ fica fixo e o desequilíbrio muda, de modo que as correntes do trecho deixam de ser proporcionais entre si. É a condição do **Corolário 2**: três correntes linearmente independentes sobre $\mathbb{C}$ tornam os 12 parâmetros identificáveis.

Diferenças em relação ao gerador original:
* só a linha 650632 recebe $\theta$ (antes o linecode `mtx601` inteiro era alterado, o que mudava também 632670, 670671 e 671680);
* os valores vêm direto da API do OpenDSS (`opendssdirect`), sem o arredondamento dos CSVs exportados;
* o sorteio de $\theta$ é o mesmo: normal com desvio relativo de 1 % e matriz simetrizada.

O código de geração está em `gerador_snapshots.py` e os arquivos do circuito em `dss/`. Requisito: `pip install opendssdirect.py` (funciona em Windows e Linux).

In [1]:
import time, sys
from pathlib import Path
import numpy as np
import pandas as pd

import gerador_snapshots as g
sys.path.insert(0, str(Path('..', '..', 'DadosBelvedere').resolve()))
import feederlib as fl

N_THETA, K, NIVEL, VAR_CARGA, SEED = 10000, 5, 0.01, 0.30, 0
N_PROC = 2                     # processos em paralelo (cada um com seu OpenDSS)
ARQ = Path(f'snapshots_{NIVEL:.2f}_K{K}.csv.gz')
RECALCULAR = False             # True: refaz os fluxos mesmo que o arquivo exista

## 1. Geração

In [2]:
if ARQ.exists() and not RECALCULAR:
    df = pd.read_csv(ARQ)
    print(f'carregado {ARQ} ({len(df)} linhas)')
else:
    t0 = time.time()
    df = g.gerar(N_THETA, K, NIVEL, VAR_CARGA, SEED, n_proc=N_PROC)
    df.to_csv(ARQ, index=False, float_format='%.10g')
    print(f'{len(df)} fluxos de potência em {time.time()-t0:.0f} s -> {ARQ}')
print('todos convergiram:', bool(df.convergiu.all()))
df.head()

carregado snapshots_0.01_K5.csv.gz (50000 linhas)
todos convergiram: True


,theta_id,snapshot,r11,r12,r22,r13,r23,r33,x11,x12,...,I2_mag_faseB,I2_ang_faseB,I1_mag_faseC,I1_ang_faseC,I2_mag_faseC,I2_ang_faseC,tap_reg1,tap_reg2,tap_reg3,convergiu
0,0,0,0.346936,0.155794,0.335692,0.159012,0.154055,0.338997,1.005019,0.498573,...,401.135844,39.399462,578.694882,94.596099,578.695379,-85.404009,1.05625,1.03750,1.05625,True
1,0,1,0.346936,0.155794,0.335692,0.159012,0.154055,0.338997,1.005019,0.498573,...,411.021277,39.641755,649.238545,91.407105,649.239100,-88.592989,1.05625,1.03750,1.06250,True
2,0,2,0.346936,0.155794,0.335692,0.159012,0.154055,0.338997,1.005019,0.498573,...,439.511221,39.116783,577.812403,92.962850,577.812930,-87.037257,1.05625,1.04375,1.05625,True
3,0,3,0.346936,0.155794,0.335692,0.159012,0.154055,0.338997,1.005019,0.498573,...,465.377493,37.261381,540.277688,94.817116,540.278182,-85.183000,1.05625,1.04375,1.05625,True
4,0,4,0.346936,0.155794,0.335692,0.159012,0.154055,0.338997,1.005019,0.498573,...,413.111318,37.940996,588.777582,92.815859,588.778112,-87.184247,1.05625,1.03750,1.05625,True


O arquivo é longo: uma linha por par (`theta_id`, `snapshot`). As colunas de medição têm os mesmos nomes do dataset original (`P1_faseA`, `V1_mag_RG60`, `I1_ang_faseA`, ...). Também há os fatores de carga (`fator_<carga>`) e os taps dos reguladores (`tap_reg1..3`), que podem mudar entre snapshots porque os reguladores continuam ativos.

In [3]:
cols_I = [f'I1_mag_fase{p}' for p in 'ABC']
resumo = pd.DataFrame({
    '|I| mínimo (A)': df[cols_I].min().values,
    '|I| máximo (A)': df[cols_I].max().values,
    'variação média de |I| entre snapshots (%)':
        (df.groupby('theta_id')[cols_I].agg(lambda s: 100 * (s.max() - s.min()) / s.mean()).mean()).values,
}, index=['fase A', 'fase B', 'fase C'])
pos = lambda r: ((df[f'tap_{r}'] - 1) / 0.00625).round().astype(int)
print('posições de tap usadas (passos de 0,625 %):', {r: sorted(pos(r).unique()) for r in ['reg1', 'reg2', 'reg3']})
print('θ com mudança de tap entre snapshots: '
      f"{100 * df.groupby('theta_id')[['tap_reg1','tap_reg2','tap_reg3']].nunique().gt(1).any(axis=1).mean():.1f} %")
resumo.round(2)

posições de tap usadas (passos de 0,625 %): {'reg1': [np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12)], 'reg2': [np.int64(6), np.int64(7), np.int64(8)], 'reg3': [np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12)]}
θ com mudança de tap entre snapshots: 99.3 %


,|I| mínimo (A),|I| máximo (A),variação média de |I| entre snapshots (%)
fase A,383.33,728.52,23.62
fase B,298.11,540.01,21.83
fase C,424.64,753.02,20.00


## 2. Verificação do Teorema 2 com os dados gerados

Com $r$ snapshots empilhados, a equação da linha $\Delta V^{(k)} = \ell\,Z\,I^{(k)}$ fornece $6r$ equações reais. O Teorema 2 prevê dimensão não observável $(3-r)(4-r)$ quando as correntes têm posto $r$: 6 para $r=1$, 2 para $r=2$ e 0 para $r\ge 3$. A corrente série desconta a pequena capacitância padrão que o OpenDSS atribui ao linecode: $I_s = I_1 - (Y/2)\,V_1$.

In [4]:
import opendssdirect as dss
dss.Text.Command(f'redirect [{g.MASTER}]'); dss.Lines.Name(g.LINHA)
C_nF = np.array(dss.Lines.CMatrix()).reshape(3, 3) * dss.Lines.Length()   # nF no trecho
Y_meio = 1j * 2 * np.pi * 60 * C_nF * 1e-9 / 2
ELL = g.COMPRIMENTO_FT / 5280                                              # milhas (θ em Ω/mi)

fas = lambda pre, suf: np.stack([fl._fasor(df[f'{pre}_mag_{s}'], df[f'{pre}_ang_{s}']) for s in suf], 1)
V1 = np.stack([fl._fasor(df[f'V{j}_mag_RG60'], df[f'V{j}_ang_RG60']) for j in (1, 2, 3)], 1)
V2 = np.stack([fl._fasor(df[f'V{j}_mag_632'], df[f'V{j}_ang_632']) for j in (1, 2, 3)], 1)
I1 = fas('I1', [f'fase{p}' for p in 'ABC'])
I2 = fas('I2', [f'fase{p}' for p in 'ABC'])
THETA = df.groupby('theta_id')[g.PARAMS].first().values
n = len(THETA)

def montar_sistema(V1, V2, I1):
    Is = I1 - V1 @ Y_meio.T
    dV = V1 - V2
    J = np.stack([fl.jacobiano(i, ELL) for i in Is]).reshape(n, K, 6, 12)
    y = np.hstack([dV.real, dV.imag]).reshape(n, K, 6)
    return J, y, Is.reshape(n, K, 3)

J, y, Is = montar_sistema(V1, V2, I1)
linhas = []
for r in range(1, K + 1):
    s = np.linalg.svd(J[:, :r].reshape(n, 6 * r, 12), compute_uv=False)
    posto = (s > 1e-9 * s[:, :1]).sum(1)
    sI = np.linalg.svd(np.swapaxes(Is[:, :r], 1, 2), compute_uv=False)
    linhas.append({'snapshots r': r, 'previsto (3-r)(4-r)': (3 - min(r, 3)) * (4 - min(r, 3)),
                   'não observável (numérico, mediana)': int(np.median(12 - posto)),
                   'amostras com a dimensão prevista (%)': 100 * np.mean(12 - posto == (3 - min(r, 3)) * (4 - min(r, 3))),
                   'σ3/σ1 das correntes (mediana)': np.median(sI[:, -1] / sI[:, 0]) if r >= 3 else np.nan})
pd.DataFrame(linhas).set_index('snapshots r').round(4)

,previsto (3-r)(4-r),"não observável (numérico, mediana)",amostras com a dimensão prevista (%),σ3/σ1 das correntes (mediana)
snapshots r,,,,
1,6,6,100.0,NaN
2,2,2,100.0,NaN
3,0,0,100.0,0.0105
4,0,0,100.0,0.0163
5,0,0,100.0,0.0194


Sem ruído, o sistema empilhado com $r \ge 3$ snapshots tem solução única. O teste abaixo resolve os mínimos quadrados por amostra e compara com o $\theta$ verdadeiro. A referência é o MAPE de simplesmente devolver o valor nominal (cerca de 0,8 × o espalhamento de 1 %).

In [5]:
TH_NOM = g.sortear_thetas(1, 0.0)[0]
mape = lambda A, B: 100 * np.mean(np.abs(A - B) / np.abs(B))

def mq(J, y, r):
    A, b = J[:, :r].reshape(n, 6 * r, 12), y[:, :r].reshape(n, 6 * r)
    return np.stack([np.linalg.lstsq(a, bb, rcond=None)[0] for a, bb in zip(A, b)])

ref = mape(np.broadcast_to(TH_NOM, THETA.shape), THETA)
tab = {f'MQ com {r} snapshots': mape(mq(J, y, r), THETA) for r in (3, 4, 5)}
tab['valor nominal (referência)'] = ref
pd.Series(tab, name='MAPE sem ruído (%)').map('{:.2e}'.format).to_frame()

,MAPE sem ruído (%)
MQ com 3 snapshots,2.09e-05
MQ com 4 snapshots,1.06e-05
MQ com 5 snapshots,7.91e-06
valor nominal (referência),8.00e-01


## 3. Prévia com ruído de medição

O mesmo estimador de mínimos quadrados (MQ) com o ruído do artigo (µPMU aleatória a cada snapshot e TIs classe 1 com erro sistemático fixo por canal, `IT_SEED = 1`) dá uma ideia do que a janela de snapshots resolve e do que ela não resolve. α escala todas as amplitudes. Esta seção é só diagnóstico: a estimação com ML fica para o notebook de estimação.

In [6]:
rng = np.random.default_rng(123)
e_ti = fl.erros_ti(1)
d = dict(V1=V1, V2=V2, I1=I1, I2=I2)
linhas = []
for alpha in (1.0, 0.1, 0.01):
    e_pmu = fl.erros_pmu(len(df), 1, rng)
    dn = fl.aplicar_ruido(d, e_pmu, e_ti, alpha)
    Jn, yn, _ = montar_sistema(dn['V1'], dn['V2'], dn['I1'])
    # mesma realização, mas sem o erro dos TIs: separa o papel do sistemático
    zero = {c: (np.zeros(3), np.zeros(3)) for c in e_ti}
    dp = fl.aplicar_ruido(d, e_pmu, zero, alpha)
    Jp, yp, _ = montar_sistema(dp['V1'], dp['V2'], dp['I1'])
    linhas.append({'α': alpha,
                   'MQ, 5 snapshots, µPMU + TI (%)': mape(mq(Jn, yn, 5), THETA),
                   'MQ, 5 snapshots, só µPMU (%)': mape(mq(Jp, yp, 5), THETA),
                   'valor nominal (%)': ref})
pd.DataFrame(linhas).set_index('α').round(3)

,"MQ, 5 snapshots, µPMU + TI (%)","MQ, 5 snapshots, só µPMU (%)",valor nominal (%)
α,,,
1.00,33.793,29.076,0.8
0.10,3.361,2.904,0.8
0.01,0.338,0.292,0.8
